In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Vision Transformers (ViT) — A Complete Tutorial

> **"An Image is Worth 16×16 Words"** — Dosovitskiy et al., 2020

This notebook is a self-contained lecture + workshop on **Vision Transformers**.
We will:

1. Build a **ViT from scratch** in PyTorch — every module explained step by step
2. Train it on **MNIST**, **Fashion-MNIST**, and **CIFAR-10** locally (runs on MacBook M4)
3. Visualise **multi-head attention maps** the way the original paper does
4. **Fine-tune a pre-trained ViT** (HuggingFace `google/vit-base-patch16-224`) on a small dataset
5. Run **zero-shot image classification** with **OpenCLIP**

---
## Table of Contents
1. [Install & Imports](#1-install--imports)
2. [ViT Architecture — every module](#2-vit-architecture--every-module)
3. [Training on MNIST](#3-training-on-mnist)
4. [Training on Fashion-MNIST](#4-training-on-fashion-mnist)
5. [Training on CIFAR-10](#5-training-on-cifar-10)
6. [Attention-Map Visualisation](#6-attention-map-visualisation)
7. [Fine-tuning a Pre-trained ViT (HuggingFace)](#7-fine-tuning-a-pre-trained-vit-huggingface)
8. [Zero-Shot Classification with OpenCLIP](#8-zero-shot-classification-with-openclip)

## 1. Install & Imports

Install the extra libraries we need once (comment out after the first run):

```bash
pip install transformers datasets open_clip_torch timm einops
```

In [ ]:
# Uncomment to install
# !pip install -q transformers datasets open_clip_torch timm einops

In [ ]:
import os, math, warnings
from pathlib import Path
from collections import defaultdict
from copy import deepcopy
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as T
from torchvision.datasets import MNIST, FashionMNIST, CIFAR10, Flowers102

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

### Device Detection

We support **CUDA** (GPU), **MPS** (Apple Silicon M-series), and **CPU** out of the box.

In [ ]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print(f"Using device: {DEVICE}")

DATA_DIR  = Path("data")
MODEL_DIR = Path("models") / "vit"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 2. ViT Architecture — Every Module

### 2.1 The Big Picture

The original **ViT** (Dosovitskiy et al., 2020) adapts the Transformer architecture — originally
designed for NLP — to work on images.  The key insight is beautifully simple:

> **Split the image into fixed-size patches, flatten each patch into a vector,
> treat the sequence of vectors exactly like a sequence of word tokens.**

```
 ┌──────────────────────────────────────────────────────┐
 │  INPUT IMAGE  (H × W × C)                           │
 └──────────────────┬───────────────────────────────────┘
                    │  Split into N patches (P × P × C each)
                    ▼
 ┌──────────────────────────────────────────────────────┐
 │  PATCH EMBEDDING  (N × D)   +  [CLS] token           │
 └──────────────────┬───────────────────────────────────┘
                    │  + Positional Encoding
                    ▼
 ┌──────────────────────────────────────────────────────┐
 │  TRANSFORMER ENCODER  ×L layers                      │
 │    each layer:  LayerNorm → MSA → residual           │
 │                 LayerNorm → MLP → residual           │
 └──────────────────┬───────────────────────────────────┘
                    │  Take [CLS] representation
                    ▼
 ┌──────────────────────────────────────────────────────┐
 │  MLP HEAD  →  class logits                           │
 └──────────────────────────────────────────────────────┘
```

Number of patches:  `N = (H/P) × (W/P)`,  sequence length = `N + 1`  (the extra 1 is `[CLS]`).

**Why a `[CLS]` token?**  Just like BERT — it acts as a global summary of the whole sequence
and is used for the final classification.  You could also average-pool all patch tokens
(called *GAP ViT*), and it often works equally well.

### 2.2 Patch Embedding

The simplest way to embed patches is a **single Conv2d** with kernel size = stride = patch size.
For a (H, W, C) image and patch size P, this produces a feature map of shape
`(C_out, H/P, W/P)` — equivalently, `N = (H/P)*(W/P)` vectors of dimension `D = C_out`.

This is identical to extracting every patch, flattening it, and multiplying by a weight matrix,
but the Conv2d implementation is faster and handles the extraction automatically.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Converts an image into a sequence of patch embeddings.

    Args:
        image_size:  side length of (square) input image
        patch_size:  side length of each (square) patch
        in_channels: number of input channels (1 for grayscale, 3 for RGB)
        embed_dim:   output embedding dimension D
    """

    def __init__(self, image_size: int, patch_size: int, in_channels: int, embed_dim: int):
        super().__init__()
        assert image_size % patch_size == 0, "Image size must be divisible by patch size"
        self.num_patches = (image_size // patch_size) ** 2
        # A single Conv2d does patch extraction + linear projection in one step
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x: Tensor) -> Tensor:
        # x: (B, C, H, W)
        x = self.proj(x)          # (B, D, H/P, W/P)
        x = x.flatten(2)          # (B, D, N)
        x = x.transpose(1, 2)     # (B, N, D)
        return x

# Quick smoke test
_pe = PatchEmbedding(image_size=28, patch_size=7, in_channels=1, embed_dim=64)
_img = torch.zeros(2, 1, 28, 28)
print("Patch embedding output:", _pe(_img).shape)  # (2, 16, 64)

### 2.3 Multi-Head Self-Attention (MSA)

Self-attention lets every patch attend to every other patch — giving ViT its global receptive
field from the very first layer (unlike CNNs which build it layer by layer).

For each head `h` out of `H` total heads, we compute:

$$\text{head}_h = \text{Softmax}\!\left(\frac{Q_h K_h^\top}{\sqrt{d_k}}\right) V_h$$

$$\text{MSA}(x) = \text{concat}(\text{head}_1, \ldots, \text{head}_H)\,W^O$$

where `d_k = D / H` is the per-head dimension.

The `attention_weights` tensor — shape `(B, H, N+1, N+1)` — is what we will visualise later.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    Standard scaled dot-product multi-head self-attention.

    Args:
        embed_dim:   total embedding dimension D
        num_heads:   number of attention heads H  (D must be divisible by H)
        attn_drop:   dropout on attention weights
        proj_drop:   dropout after output projection
    """

    def __init__(self, embed_dim: int, num_heads: int,
                 attn_drop: float = 0.0, proj_drop: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5   # 1/sqrt(d_k)

        # Project input to Q, K, V all at once
        self.qkv  = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)

        # Storage hook for visualisation
        self.last_attn_weights: Optional[Tensor] = None

    def forward(self, x: Tensor) -> Tensor:
        B, N, D = x.shape
        H, d = self.num_heads, self.head_dim

        # (B, N, 3D) → (B, N, 3, H, d) → (3, B, H, N, d)
        qkv = self.qkv(x).reshape(B, N, 3, H, d).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)          # each: (B, H, N, d)

        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, H, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # Save for later visualisation (detached to avoid holding gradients)
        self.last_attn_weights = attn.detach()

        x = (attn @ v).transpose(1, 2).reshape(B, N, D)  # merge heads
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

### 2.4 MLP Block

After attention, each position is processed independently by a small **two-layer MLP**
with a GELU activation.  The hidden dimension is typically `4 × D` (controlled by `mlp_ratio`).

In [ ]:
class MLPBlock(nn.Module):
    """Position-wise feed-forward network used inside each Transformer block."""

    def __init__(self, embed_dim: int, mlp_ratio: float = 4.0, drop: float = 0.0):
        super().__init__()
        hidden = int(embed_dim * mlp_ratio)
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(drop),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

### 2.5 Transformer Encoder Block

One encoder block = **LayerNorm → MSA → residual** then **LayerNorm → MLP → residual**.

Pre-norm (norm *before* the sub-layer) is the standard ViT choice — it is more stable
to train than the original post-norm Transformer.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Single Transformer encoder layer (pre-norm variant)."""

    def __init__(self, embed_dim: int, num_heads: int,
                 mlp_ratio: float = 4.0,
                 attn_drop: float = 0.0, drop: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, attn_drop, drop)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio, drop)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

### 2.6 Full Vision Transformer

Putting it all together:

1. **PatchEmbedding** converts the image to `(B, N, D)`
2. A learnable `[CLS]` token is prepended → `(B, N+1, D)`
3. Learnable **positional embeddings** are added (same shape — the model learns spatial structure)
4. `L` **TransformerEncoderBlocks** process the sequence
5. The `[CLS]` token's output is passed through a **classification head**

The class token receives gradient signal from every patch token through cross-attention
and therefore summarises the whole image.

> **Positional encoding is crucial** — without it, ViT is a *bag of patches* and has no idea
> about spatial layout.  Unlike NLP which often uses sinusoidal encodings, ViT uses learnable
> 1D position embeddings; experiments in the paper showed this works just as well.

In [ ]:
class VisionTransformer(nn.Module):
    """
    Minimal Vision Transformer for image classification.

    Args:
        image_size:  H = W of input image
        patch_size:  size of each patch (image_size must be divisible by this)
        in_channels: 1 (grayscale) or 3 (RGB)
        num_classes: output classes
        embed_dim:   token / embedding dimension D
        num_heads:   attention heads per layer
        num_layers:  number of Transformer encoder blocks
        mlp_ratio:   MLP hidden dim = mlp_ratio × embed_dim
        attn_drop:   attention dropout rate
        drop_rate:   general dropout rate
    """

    def __init__(
        self,
        image_size: int   = 28,
        patch_size: int   = 7,
        in_channels: int  = 1,
        num_classes: int  = 10,
        embed_dim: int    = 64,
        num_heads: int    = 4,
        num_layers: int   = 4,
        mlp_ratio: float  = 4.0,
        attn_drop: float  = 0.0,
        drop_rate: float  = 0.0,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        N = self.patch_embed.num_patches

        # Learnable [CLS] token and positional embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, N + 1, embed_dim))
        self.pos_drop  = nn.Dropout(drop_rate)

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, attn_drop, drop_rate)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        # Initialise position embeddings and CLS token
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: Tensor) -> Tensor:
        B = x.shape[0]
        x = self.patch_embed(x)                          # (B, N, D)

        cls = self.cls_token.expand(B, -1, -1)           # (B, 1, D)
        x   = torch.cat([cls, x], dim=1)                 # (B, N+1, D)
        x   = self.pos_drop(x + self.pos_embed)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)
        return self.head(x[:, 0])                        # CLS token → logits

    def get_attention_weights(self, layer: int = -1) -> Optional[Tensor]:
        """Return attention weights saved during the last forward pass."""
        return self.blocks[layer].attn.last_attn_weights

# ── Quick sanity check ─────────────────────────────────────────────────────────
_vit = VisionTransformer()
_x   = torch.zeros(2, 1, 28, 28)
print("ViT output shape:", _vit(_x).shape)   # (2, 10)

total_params = sum(p.numel() for p in _vit.parameters())
print(f"Total parameters: {total_params:,}")

### 2.7 Training & Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds   = model(imgs).argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
    return correct / total


def train(model, train_loader, val_loader, epochs, lr, device, label=""):
    """Full training loop with cosine LR schedule."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # Cosine annealing over total steps
    total_steps = epochs * len(train_loader)
    scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    history = defaultdict(list)
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_acc = evaluate(model, val_loader, device)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)
        if epoch % max(1, epochs // 5) == 0 or epoch == 1:
            print(f"{label}  Epoch {epoch:3d}/{epochs} | "
                  f"loss {tr_loss:.4f} | train acc {tr_acc:.3f} | val acc {val_acc:.3f}")
    return history


def plot_history(histories: dict, title="Training History"):
    """Plot training curves for one or more runs."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        axes[0].plot(h["train_loss"], label=name)
        axes[1].plot(h["val_acc"],    label=name)
    axes[0].set_title("Train Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].set_title("Val Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

## 3. Training on MNIST

MNIST images are **28 × 28 grayscale**.  We use `patch_size=7`, giving `4 × 4 = 16` patches.

| Hyper-parameter | Value |
|---|---|
| image size | 28 × 28 |
| patch size | 7 × 7 |
| num patches | 16 |
| embed dim | 128 |
| heads | 4 |
| layers | 4 |
| parameters | ~500K |

This tiny model trains to **>98 % accuracy** on MNIST in a few minutes on any modern machine.

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────────────
MNIST_MEAN, MNIST_STD = (0.1307,), (0.3081,)

mnist_train_tf = T.Compose([T.RandomAffine(degrees=10, translate=(0.1, 0.1)),
                             T.ToTensor(),
                             T.Normalize(MNIST_MEAN, MNIST_STD)])
mnist_test_tf  = T.Compose([T.ToTensor(), T.Normalize(MNIST_MEAN, MNIST_STD)])

mnist_train = MNIST(DATA_DIR, train=True,  download=True, transform=mnist_train_tf)
mnist_test  = MNIST(DATA_DIR, train=False, download=True, transform=mnist_test_tf)

mnist_train_loader = DataLoader(mnist_train, batch_size=256, shuffle=True,  num_workers=0)
mnist_test_loader  = DataLoader(mnist_test,  batch_size=512, shuffle=False, num_workers=0)

print(f"MNIST train: {len(mnist_train):,}  |  test: {len(mnist_test):,}")

In [ ]:
# ── Visualise a few samples ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flatten()):
    img, label = mnist_train[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(str(label), fontsize=9)
    ax.axis("off")
plt.suptitle("MNIST samples", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────────
mnist_vit = VisionTransformer(
    image_size  = 28,
    patch_size  = 7,
    in_channels = 1,
    num_classes = 10,
    embed_dim   = 128,
    num_heads   = 4,
    num_layers  = 4,
    mlp_ratio   = 4.0,
    attn_drop   = 0.1,
    drop_rate   = 0.1,
).to(DEVICE)

n_params = sum(p.numel() for p in mnist_vit.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────────
mnist_history = train(
    model        = mnist_vit,
    train_loader = mnist_train_loader,
    val_loader   = mnist_test_loader,
    epochs       = 15,
    lr           = 3e-4,
    device       = DEVICE,
    label        = "[MNIST]",
)

plot_history({"MNIST ViT": mnist_history}, title="ViT on MNIST")

In [ ]:
final_mnist_acc = evaluate(mnist_vit, mnist_test_loader, DEVICE)
print(f"Final MNIST test accuracy: {final_mnist_acc:.4f}  ({final_mnist_acc*100:.2f}%)")

## 4. Training on Fashion-MNIST

**Fashion-MNIST** has the same 28 × 28 shape but replaces digits with 10 clothing categories
(T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle Boot).

It is a harder dataset — a simple CNN baseline gets ~92 %, state-of-the-art is ~96 %.
We reuse the exact same architecture (same patch layout, same hyper-parameters) to show
that ViT generalises well across different visual domains without architecture changes.

**What to observe:**
- Similar convergence speed to MNIST
- Lower absolute accuracy (harder task)
- Attention maps will focus on the *shape* of clothing items

In [ ]:
FM_MEAN, FM_STD = (0.2860,), (0.3530,)

fm_train_tf = T.Compose([T.RandomHorizontalFlip(),
                          T.RandomAffine(degrees=10, translate=(0.1, 0.1)),
                          T.ToTensor(),
                          T.Normalize(FM_MEAN, FM_STD)])
fm_test_tf  = T.Compose([T.ToTensor(), T.Normalize(FM_MEAN, FM_STD)])

fm_train = FashionMNIST(DATA_DIR, train=True,  download=True, transform=fm_train_tf)
fm_test  = FashionMNIST(DATA_DIR, train=False, download=True, transform=fm_test_tf)

fm_train_loader = DataLoader(fm_train, batch_size=256, shuffle=True,  num_workers=0)
fm_test_loader  = DataLoader(fm_test,  batch_size=512, shuffle=False, num_workers=0)

FM_CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
              "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"]

# Visualise
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flatten()):
    img, label = fm_train[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(FM_CLASSES[label], fontsize=7)
    ax.axis("off")
plt.suptitle("Fashion-MNIST samples", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
fm_vit = VisionTransformer(
    image_size=28, patch_size=7, in_channels=1, num_classes=10,
    embed_dim=128, num_heads=4, num_layers=4, mlp_ratio=4.0,
    attn_drop=0.1, drop_rate=0.1,
).to(DEVICE)

fm_history = train(
    model=fm_vit, train_loader=fm_train_loader, val_loader=fm_test_loader,
    epochs=20, lr=3e-4, device=DEVICE, label="[FashionMNIST]",
)

final_fm_acc = evaluate(fm_vit, fm_test_loader, DEVICE)
print(f"Final Fashion-MNIST test accuracy: {final_fm_acc:.4f}  ({final_fm_acc*100:.2f}%)")
plot_history({"Fashion-MNIST ViT": fm_history}, title="ViT on Fashion-MNIST")

## 5. Training on CIFAR-10

CIFAR-10 images are **32 × 32 RGB** — three colour channels and a more complex, natural-image
distribution.  We need to scale up slightly:

| Change from MNIST config | Reason |
|---|---|
| `patch_size = 4` | Finer granularity on 32 px; gives 64 patches |
| `embed_dim = 256` | More capacity for colour / texture patterns |
| `num_heads = 8` | More heads to specialise on different features |
| `num_layers = 6` | Deeper model for harder task |
| Stronger augmentation | RandomCrop, ColorJitter |

Training ViT from scratch on CIFAR-10 is notoriously tricky — ViT prefers more data.
We add **CutOut** regularisation and a warm-up LR schedule to stabilise training.
Typical from-scratch accuracy: **75–80 %** (compare: ResNet-18 ~93 %).

> **Key insight:** This is exactly why pre-trained ViT (Section 7) is so valuable.
> Pre-training on ImageNet-21k gives the model rich low-level representations that
> transfer immediately to small datasets.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

cifar_train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.2)),   # CutOut-style regularisation
])
cifar_test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

cifar_train = CIFAR10(DATA_DIR, train=True,  download=True, transform=cifar_train_tf)
cifar_test  = CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_test_tf)

cifar_train_loader = DataLoader(cifar_train, batch_size=256, shuffle=True,  num_workers=0)
cifar_test_loader  = DataLoader(cifar_test,  batch_size=512, shuffle=False, num_workers=0)

CIFAR_CLASSES = ["airplane","automobile","bird","cat","deer",
                 "dog","frog","horse","ship","truck"]

# Visualise (un-normalise for display)
inv_norm = T.Normalize(
    mean=[-m/s for m, s in zip(CIFAR_MEAN, CIFAR_STD)],
    std=[1/s for s in CIFAR_STD])

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
_raw_cifar = CIFAR10(DATA_DIR, train=True, download=False,
                     transform=T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)]))
for i, ax in enumerate(axes.flatten()):
    img, label = _raw_cifar[i]
    img_disp   = inv_norm(img).permute(1, 2, 0).clip(0, 1)
    ax.imshow(img_disp)
    ax.set_title(CIFAR_CLASSES[label], fontsize=7)
    ax.axis("off")
plt.suptitle("CIFAR-10 samples", y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
cifar_vit = VisionTransformer(
    image_size  = 32,
    patch_size  = 4,
    in_channels = 3,
    num_classes = 10,
    embed_dim   = 256,
    num_heads   = 8,
    num_layers  = 6,
    mlp_ratio   = 4.0,
    attn_drop   = 0.1,
    drop_rate   = 0.1,
).to(DEVICE)

n_params = sum(p.numel() for p in cifar_vit.parameters() if p.requires_grad)
print(f"CIFAR ViT trainable parameters: {n_params:,}")

In [ ]:
# Warm-up + cosine schedule implemented manually for finer control

def train_cifar(model, train_loader, val_loader, epochs, lr_max, warmup_epochs, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=5e-4)
    history   = defaultdict(list)

    steps_per_epoch = len(train_loader)
    warmup_steps    = warmup_epochs * steps_per_epoch
    total_steps     = epochs * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)           # linear warm-up
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress)) # cosine decay

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,
                    lr_lambda=lambda s: lr_lambda(s) * lr_max / 1e-6)

    global_step = 0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(imgs), labels, label_smoothing=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            global_step += 1
            total_loss += loss.item() * imgs.size(0)
            correct    += (model(imgs).argmax(1) == labels).sum().item()
            total      += imgs.size(0)
        val_acc = evaluate(model, val_loader, device)
        tr_acc  = correct / total
        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)
        if epoch % max(1, epochs // 5) == 0 or epoch == 1:
            print(f"[CIFAR-10]  Epoch {epoch:3d}/{epochs} | "
                  f"loss {total_loss/total:.4f} | train {tr_acc:.3f} | val {val_acc:.3f} | "
                  f"lr {scheduler.get_last_lr()[0]:.2e}")
    return history

cifar_history = train_cifar(
    model=cifar_vit, train_loader=cifar_train_loader, val_loader=cifar_test_loader,
    epochs=30, lr_max=3e-4, warmup_epochs=5, device=DEVICE,
)
final_cifar_acc = evaluate(cifar_vit, cifar_test_loader, DEVICE)
print(f"\nFinal CIFAR-10 test accuracy: {final_cifar_acc:.4f}  ({final_cifar_acc*100:.2f}%)")
plot_history({"CIFAR-10 ViT": cifar_history}, title="ViT from Scratch on CIFAR-10")

## 6. Attention-Map Visualisation

### What the original paper shows

Figure 6 of *"An Image is Worth 16×16 Words"* overlays the attention weights of the `[CLS]` token
on the input image.  Concretely:

- After a forward pass, each head in the **last Transformer block** produces an attention matrix
  of shape `(N+1, N+1)`.
- The row corresponding to `[CLS]` (row 0) shows which patches the token attends to.
- We extract that row, drop the `[CLS]`→`[CLS]` entry, reshape `N → (H/P, W/P)`, and upsample
  back to the original image size.

### Attention Rollout

A more principled approach is **Attention Rollout** (Abnar & Zuidema, 2020):
instead of using only the last layer, we *roll* attention through all layers by multiplying
them together, accounting for residual connections.  This captures how information flows
from early patches all the way to the `[CLS]` token.

In [ ]:
def cls_attention_map(model: VisionTransformer, img_tensor: Tensor,
                      layer: int = -1) -> np.ndarray:
    """
    Run a forward pass and return the CLS-token attention map as a 2-D numpy array
    (normalised to [0, 1]) at the original image resolution.

    img_tensor: (1, C, H, W) on any device
    """
    model.eval()
    with torch.no_grad():
        _ = model(img_tensor)
    attn = model.get_attention_weights(layer)  # (1, H, N+1, N+1)
    attn = attn.mean(dim=1)                    # average over heads → (1, N+1, N+1)
    # CLS attends to patches: row 0, columns 1: (skip CLS→CLS)
    cls_attn = attn[0, 0, 1:]                  # (N,)

    # Reshape to grid
    ph = pw = int(cls_attn.shape[0] ** 0.5)
    attn_map = cls_attn.reshape(ph, pw).cpu().numpy()

    # Upsample to image size
    patch_size = img_tensor.shape[-1] // ph
    attn_map   = np.kron(attn_map, np.ones((patch_size, patch_size)))  # nearest-neighbour up
    attn_map   = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    return attn_map


def attention_rollout(model: VisionTransformer, img_tensor: Tensor) -> np.ndarray:
    """
    Attention Rollout across all Transformer layers.
    Returns a normalised attention map at image resolution.
    """
    model.eval()
    with torch.no_grad():
        _ = model(img_tensor)

    rollout = None
    for block in model.blocks:
        attn = block.attn.last_attn_weights   # (1, H, N+1, N+1)
        attn = attn.mean(dim=1)[0]            # (N+1, N+1)  — average heads
        # Residual connection: 0.5 * A + 0.5 * I
        I    = torch.eye(attn.size(0), device=attn.device)
        attn = 0.5 * attn + 0.5 * I
        attn = attn / attn.sum(dim=-1, keepdim=True)  # re-normalise rows
        rollout = attn if rollout is None else attn @ rollout

    cls_attn = rollout[0, 1:]          # (N,)
    ph = pw  = int(cls_attn.shape[0] ** 0.5)
    attn_map = cls_attn.reshape(ph, pw).cpu().numpy()
    patch_size = img_tensor.shape[-1] // ph
    attn_map   = np.kron(attn_map, np.ones((patch_size, patch_size)))
    attn_map   = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    return attn_map

In [ ]:
def show_attention(model, dataset, inv_transform, classes, n=6, cmap="inferno",
                   title_prefix="", device=DEVICE, is_grayscale=False):
    """Display raw image, last-layer CLS attention, and attention rollout side by side."""
    fig, axes = plt.subplots(n, 3, figsize=(10, 2.8 * n))
    cols = ["Input", "CLS Attention (last layer)", "Attention Rollout"]
    for ax, c in zip(axes[0], cols):
        ax.set_title(c, fontsize=11, fontweight="bold")

    idxs = np.random.choice(len(dataset), n, replace=False)
    for row, idx in enumerate(idxs):
        img_raw, label = dataset[idx]
        img_t = img_raw.unsqueeze(0).to(device)

        # Display image
        if is_grayscale:
            img_disp = img_raw.squeeze().numpy()
            axes[row, 0].imshow(img_disp, cmap="gray")
        else:
            img_disp = inv_transform(img_raw).permute(1, 2, 0).numpy().clip(0, 1)
            axes[row, 0].imshow(img_disp)
        axes[row, 0].set_ylabel(classes[label], rotation=0, labelpad=50,
                                va="center", fontsize=10)

        cls_map  = cls_attention_map(model, img_t)
        roll_map = attention_rollout(model, img_t)

        for col, amap in enumerate([cls_map, roll_map], start=1):
            axes[row, col].imshow(amap, cmap=cmap)
            # Overlay semi-transparent original
            if is_grayscale:
                axes[row, col].imshow(img_disp, cmap="gray", alpha=0.35)
            else:
                axes[row, col].imshow(img_disp, alpha=0.35)

        for ax in axes[row]:
            ax.axis("off")

    plt.suptitle(title_prefix + " — Attention Maps", fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

#### MNIST Attention Maps

Each head learns different aspects of the digit.  In the last layer the `[CLS]` token
typically focuses on stroke regions — you can see it "reading" the digit shape.

In [ ]:
show_attention(
    model        = mnist_vit,
    dataset      = MNIST(DATA_DIR, train=False, download=False, transform=mnist_test_tf),
    inv_transform= lambda x: x,
    classes      = [str(i) for i in range(10)],
    n            = 6,
    title_prefix = "MNIST",
    is_grayscale = True,
)

#### Fashion-MNIST Attention Maps

For clothing items, attention often focuses on the *silhouette* and distinctive parts
(collar, heel, strap).  Compare how different heads specialise.

In [ ]:
show_attention(
    model        = fm_vit,
    dataset      = FashionMNIST(DATA_DIR, train=False, download=False, transform=fm_test_tf),
    inv_transform= lambda x: x,
    classes      = FM_CLASSES,
    n            = 6,
    title_prefix = "Fashion-MNIST",
    is_grayscale = True,
)

#### CIFAR-10 Attention Maps

With colour images and more patches (64 vs 16), the spatial structure is richer.
Notice that the rollout map gives a smoother, more coherent region than the last-layer map alone.

In [ ]:
cifar_inv_norm = T.Normalize(
    mean=[-m/s for m, s in zip(CIFAR_MEAN, CIFAR_STD)],
    std=[1/s for s in CIFAR_STD])

show_attention(
    model        = cifar_vit,
    dataset      = CIFAR10(DATA_DIR, train=False, download=False, transform=cifar_test_tf),
    inv_transform= cifar_inv_norm,
    classes      = CIFAR_CLASSES,
    n            = 6,
    title_prefix = "CIFAR-10",
    is_grayscale = False,
)

#### Per-Head Attention Comparison

Let's visualise all individual heads of the last layer to see how they specialise.

In [ ]:
def show_all_heads(model: VisionTransformer, img_tensor: Tensor, img_disp: np.ndarray,
                   label_str: str, is_grayscale: bool = False):
    """Show each attention head separately for one image."""
    model.eval()
    with torch.no_grad():
        _ = model(img_tensor)
    attn = model.get_attention_weights(-1)  # (1, H, N+1, N+1)
    H    = attn.shape[1]

    fig, axes = plt.subplots(2, H // 2, figsize=(H * 1.8, 4))
    axes = axes.flatten()
    for h, ax in enumerate(axes):
        head_attn = attn[0, h, 0, 1:].cpu().numpy()   # CLS row, skip CLS col
        ph = int(head_attn.shape[0] ** 0.5)
        amap = head_attn.reshape(ph, ph)
        patch_size = img_tensor.shape[-1] // ph
        amap = np.kron(amap, np.ones((patch_size, patch_size)))
        amap = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
        ax.imshow(amap, cmap="inferno")
        if is_grayscale:
            ax.imshow(img_disp, cmap="gray", alpha=0.35)
        else:
            ax.imshow(img_disp, alpha=0.35)
        ax.set_title(f"Head {h}", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"All attention heads — last layer | Label: {label_str}", fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

# Pick one CIFAR-10 test image
_cifar_test_raw = CIFAR10(DATA_DIR, train=False, download=False, transform=cifar_test_tf)
_idx = 7
_img_t, _label = _cifar_test_raw[_idx]
_img_disp = cifar_inv_norm(_img_t).permute(1, 2, 0).numpy().clip(0, 1)

show_all_heads(cifar_vit, _img_t.unsqueeze(0).to(DEVICE), _img_disp,
               label_str=CIFAR_CLASSES[_label], is_grayscale=False)

## 7. Fine-tuning a Pre-trained ViT (HuggingFace)

### Why fine-tune instead of training from scratch?

The from-scratch CIFAR-10 result (~75–80 %) is significantly below CNN baselines (~93 %).
The reason is well understood: **ViT has no inductive biases** (no translation invariance,
no local convolutions) — it must learn them from data.  With only 50k training images,
it cannot learn them reliably.

**Solution:** use a model pre-trained on a much larger corpus.
We will use `google/vit-base-patch16-224`:
- Pre-trained on **ImageNet-21k** (~14 M images, 21k classes)
- Then fine-tuned on **ImageNet-1k** (1.28 M images, 1k classes)
- Architecture: ViT-B/16 — 12 layers, 12 heads, 768-dim, 16×16 patches

Fine-tuning strategy:
1. Replace the final classification head with a new `nn.Linear(768, num_classes)`
2. **Freeze** all layers except the head and last 2 Transformer blocks (parameter-efficient)
3. Use a small LR (1e-4) and cosine schedule
4. Resize CIFAR-10 images from 32 → 224 px (required by the model's patch tokeniser)

> **Note on Meta EUPE and TIPSv2:** These are large-scale internal Meta datasets not publicly
> released.  The HuggingFace Hub hosts many excellent open alternatives including
> `facebook/dinov2-base` (DINOv2, trained on LVD-142M) and `openai/clip-vit-large-patch14`.
> The fine-tuning code below works identically for any of them — just swap the model ID.

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor

MODEL_ID  = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(MODEL_ID)
print("Expected input size:", processor.size)

In [ ]:
# Build a HuggingFace-compatible transform using the processor's config
hf_mean = processor.image_mean
hf_std  = processor.image_std
img_sz  = processor.size["height"]   # 224

hf_train_tf = T.Compose([
    T.Resize((img_sz, img_sz)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(hf_mean, hf_std),
])
hf_test_tf  = T.Compose([
    T.Resize((img_sz, img_sz)),
    T.ToTensor(),
    T.Normalize(hf_mean, hf_std),
])

# For speed we use only a 5k subset of CIFAR-10 for fine-tuning
#   (demonstrates the data-efficiency advantage of pre-training)
_full_train = CIFAR10(DATA_DIR, train=True,  download=False, transform=hf_train_tf)
_full_test  = CIFAR10(DATA_DIR, train=False, download=False, transform=hf_test_tf)

rng     = np.random.default_rng(42)
sub_idx = rng.choice(len(_full_train), size=5_000, replace=False)
hf_train_ds = Subset(_full_train, sub_idx)
hf_test_ds  = _full_test          # keep full 10k test set

hf_train_loader = DataLoader(hf_train_ds, batch_size=32, shuffle=True,  num_workers=0)
hf_test_loader  = DataLoader(hf_test_ds,  batch_size=64, shuffle=False, num_workers=0)

print(f"Fine-tune train: {len(hf_train_ds):,}  |  test: {len(hf_test_ds):,}")

In [ ]:
# Load pre-trained ViT and replace the classification head
hf_vit = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels          = 10,
    ignore_mismatched_sizes = True,   # head shape changed: 1000 → 10
)

# Freeze everything except: the new classification head + last 2 transformer blocks
def freeze_for_finetuning(model, unfreeze_last_n_blocks: int = 2):
    for param in model.parameters():
        param.requires_grad = False
    # Un-freeze classifier head
    for param in model.classifier.parameters():
        param.requires_grad = True
    # Un-freeze last N encoder blocks
    total_blocks = len(model.vit.encoder.layer)
    for block in model.vit.encoder.layer[total_blocks - unfreeze_last_n_blocks:]:
        for param in block.parameters():
            param.requires_grad = True
    # Un-freeze final LayerNorm
    for param in model.vit.layernorm.parameters():
        param.requires_grad = True

freeze_for_finetuning(hf_vit, unfreeze_last_n_blocks=2)

trainable = sum(p.numel() for p in hf_vit.parameters() if p.requires_grad)
total     = sum(p.numel() for p in hf_vit.parameters())
print(f"Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)")

hf_vit = hf_vit.to(DEVICE)

In [ ]:
def train_hf_vit(model, train_loader, test_loader, epochs, lr, device):
    """Fine-tune HuggingFace ViT.  HF models return a dict; we pull .logits."""
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader))
    history   = defaultdict(list)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out    = model(pixel_values=imgs)
            loss   = F.cross_entropy(out.logits, labels, label_smoothing=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.logits.argmax(1) == labels).sum().item()
            total      += imgs.size(0)

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                val_correct += (model(pixel_values=imgs).logits.argmax(1) == labels).sum().item()
                val_total   += imgs.size(0)

        tr_acc  = correct / total
        val_acc = val_correct / val_total
        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)
        print(f"[HF ViT FT]  Epoch {epoch:2d}/{epochs} | "
              f"loss {total_loss/total:.4f} | train {tr_acc:.3f} | val {val_acc:.3f}")
    return history

hf_history = train_hf_vit(
    model=hf_vit, train_loader=hf_train_loader, test_loader=hf_test_loader,
    epochs=10, lr=5e-5, device=DEVICE,
)

In [ ]:
plot_history({"Pre-trained ViT (5k labels)": hf_history}, title="Fine-tuned ViT on CIFAR-10")

# Compare
print("\n=== Accuracy Comparison ===")
print(f"  ViT from scratch (50k labels): {final_cifar_acc:.3f}")
hf_final = hf_history['val_acc'][-1]
print(f"  Fine-tuned ViT  ( 5k labels): {hf_final:.3f}")
print(f"  Fine-tuning uses {50000//5000}× less labelled data yet achieves higher accuracy!")

### Visualising Attention from the Pre-trained ViT

HuggingFace ViT stores attention in a slightly different way.
We hook into the model's attention modules to extract the maps.

In [ ]:
def get_hf_attention_maps(model, img_tensor: Tensor):
    """
    Extract last-layer CLS attention from a HuggingFace ViTForImageClassification.
    Requires output_attentions=True.
    """
    model.eval()
    with torch.no_grad():
        out = model(pixel_values=img_tensor, output_attentions=True)
    # out.attentions: tuple of (B, H, N+1, N+1) per layer
    last_attn = out.attentions[-1]          # (1, H, N+1, N+1)
    cls_attn  = last_attn[0].mean(0)[0, 1:] # average heads → CLS row → patches
    ph = int(cls_attn.shape[0] ** 0.5)
    amap = cls_attn.reshape(ph, ph).cpu().numpy()
    patch_size = img_tensor.shape[-1] // ph
    amap = np.kron(amap, np.ones((patch_size, patch_size)))
    amap = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
    return amap

# Visualise on a few CIFAR-10 test images
hf_inv = T.Normalize(mean=[-m/s for m,s in zip(hf_mean, hf_std)], std=[1/s for s in hf_std])

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row in range(3):
    img_t, label = _full_test[row * 300]
    img_disp = hf_inv(img_t).permute(1, 2, 0).numpy().clip(0, 1)
    amap = get_hf_attention_maps(hf_vit, img_t.unsqueeze(0).to(DEVICE))

    axes[row, 0].imshow(img_disp); axes[row, 0].set_title(CIFAR_CLASSES[label])
    axes[row, 1].imshow(amap, cmap="inferno"); axes[row, 1].set_title("CLS Attention")
    axes[row, 2].imshow(img_disp); axes[row, 2].imshow(amap, cmap="inferno", alpha=0.5)
    axes[row, 2].set_title("Overlay")
    for ax in axes[row]: ax.axis("off")

plt.suptitle("Pre-trained ViT attention maps (CIFAR-10, resized to 224)", fontsize=13)
plt.tight_layout(); plt.show()

## 8. Zero-Shot Classification with OpenCLIP

### What is CLIP?

**CLIP** (Contrastive Language–Image Pre-Training, Radford et al. 2021) jointly trains a
**vision encoder** (ViT) and a **text encoder** (Transformer) to map images and their
descriptions close together in a shared embedding space.

Training objective:
- Given a batch of (image, caption) pairs, maximise cosine similarity of paired embeddings
  while minimising cross-pair similarity (contrastive loss on a matrix of `batch × batch` pairs).

**Zero-shot classification** with CLIP:
1. Embed the image → vector `v_img`
2. For each class, embed the text prompt `"a photo of a {class}"` → `v_text_k`
3. Classify as `argmax_k  cosine_sim(v_img, v_text_k)`

No labels needed at inference — the model generalises through natural language.

**OpenCLIP** is an open-source reproduction of CLIP trained on the public **LAION-2B** dataset
(2 billion image–text pairs from the internet), achieving results comparable to or better
than the original OpenAI CLIP.

In [ ]:
import open_clip

# List available models (first time: downloads weights, ~350 MB)
# open_clip.list_pretrained()  # uncomment to browse all options

model_name     = "ViT-B-32"
pretrained_tag = "laion2b_s34b_b79k"    # best open LAION-2B checkpoint for ViT-B/32

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    model_name, pretrained=pretrained_tag)
clip_tokenizer = open_clip.get_tokenizer(model_name)

clip_model = clip_model.to(DEVICE).eval()
print(f"OpenCLIP {model_name} ({pretrained_tag}) loaded")
print(f"Parameters: {sum(p.numel() for p in clip_model.parameters()):,}")

In [ ]:
# ── Build text embeddings for CIFAR-10 class prompts ────────────────────────
PROMPT_TEMPLATES = [
    "a photo of a {}.",
    "a high-resolution photo of a {}.",
    "a blurry photo of a {}.",
    "a photograph of a {}.",
    "a picture of a {}.",
]

@torch.no_grad()
def embed_text_prompts(model, tokenizer, classes, templates, device):
    """Embed each class with multiple prompt templates and average the embeddings."""
    class_embeddings = []
    for cls in classes:
        texts = tokenizer([t.format(cls) for t in templates]).to(device)
        embs  = model.encode_text(texts)
        embs  = embs / embs.norm(dim=-1, keepdim=True)   # L2 normalise
        class_embeddings.append(embs.mean(dim=0))         # template ensemble
    class_embeddings = torch.stack(class_embeddings)      # (C, D)
    return class_embeddings / class_embeddings.norm(dim=-1, keepdim=True)

text_embs = embed_text_prompts(clip_model, clip_tokenizer,
                                CIFAR_CLASSES, PROMPT_TEMPLATES, DEVICE)
print("Text embedding matrix shape:", text_embs.shape)   # (10, 512)

In [ ]:
@torch.no_grad()
def zero_shot_accuracy(model, dataset, text_embs, preprocess, device, n_samples=None):
    """
    Evaluate zero-shot CLIP on a dataset.
    dataset items must be (PIL.Image, label).
    We pass each image through preprocess (CLIP's standard transform) individually.
    """
    # Build a loader using CLIP's preprocessing
    clip_dataset = CIFAR10(DATA_DIR, train=False, download=False,
                           transform=preprocess)
    loader = DataLoader(clip_dataset, batch_size=128, shuffle=False, num_workers=0)

    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        img_embs = model.encode_image(imgs)
        img_embs = img_embs / img_embs.norm(dim=-1, keepdim=True)
        logits   = img_embs @ text_embs.T * model.logit_scale.exp()
        preds    = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        if n_samples and total >= n_samples:
            break
    return correct / total

clip_acc = zero_shot_accuracy(clip_model, None, text_embs, clip_preprocess, DEVICE)
print(f"OpenCLIP zero-shot CIFAR-10 accuracy: {clip_acc:.4f}  ({clip_acc*100:.2f}%)")

In [ ]:
# ── Qualitative: predict labels for individual images ──────────────────────────
@torch.no_grad()
def predict_single(model, image: Image.Image, text_embs: Tensor, classes, preprocess, device, top_k=5):
    img_t    = preprocess(image).unsqueeze(0).to(device)
    img_emb  = model.encode_image(img_t)
    img_emb  = img_emb / img_emb.norm(dim=-1, keepdim=True)
    logits   = (img_emb @ text_embs.T * model.logit_scale.exp()).squeeze(0)
    probs    = logits.softmax(dim=-1).cpu().numpy()
    top_idxs = probs.argsort()[::-1][:top_k]
    return [(classes[i], probs[i]) for i in top_idxs]

# Sample CIFAR-10 PIL images (CIFAR-10 stores raw PIL internally)
_pil_cifar = CIFAR10(DATA_DIR, train=False, download=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
sample_idxs = [0, 50, 200, 350, 500, 750, 999, 1200]

for ax_col, idx in zip(axes.T, sample_idxs):
    pil_img, true_label = _pil_cifar[idx]
    preds = predict_single(clip_model, pil_img, text_embs, CIFAR_CLASSES,
                           clip_preprocess, DEVICE)

    ax_col[0].imshow(pil_img)
    ax_col[0].set_title(f"GT: {CIFAR_CLASSES[true_label]}", fontsize=9, color="green")
    ax_col[0].axis("off")

    # Bar chart of probabilities
    names  = [p[0] for p in preds]
    scores = [p[1] for p in preds]
    colors = ["#2ecc71" if n == CIFAR_CLASSES[true_label] else "#3498db" for n in names]
    ax_col[1].barh(names[::-1], scores[::-1], color=colors[::-1])
    ax_col[1].set_xlim(0, 1)
    ax_col[1].set_xlabel("P", fontsize=8)
    ax_col[1].tick_params(labelsize=7)

plt.suptitle("OpenCLIP ViT-B/32 Zero-Shot Predictions on CIFAR-10", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

### Prompt Engineering Effect

One interesting property of CLIP-style models is sensitivity to the text prompt.
Let's compare different prompt styles:

In [ ]:
prompt_variants = {
    "Simple":     ["{}."],
    "Photo":      ["a photo of a {}."],
    "Descriptive":["a high-quality photo of a {}, a type of {}."],
    "Ensemble":   PROMPT_TEMPLATES,   # 5 templates averaged
}

print("Prompt engineering effect on CIFAR-10 zero-shot accuracy:")
print("-" * 50)
for name, templates in prompt_variants.items():
    temb = embed_text_prompts(clip_model, clip_tokenizer, CIFAR_CLASSES, templates, DEVICE)
    acc  = zero_shot_accuracy(clip_model, None, temb, clip_preprocess, DEVICE)
    bar  = "#" * int(acc * 40)
    print(f"  {name:<12}: {acc:.3f}  {bar}")

#### Embedding Space Similarity

CLIP's joint embedding space is interesting to explore.  Let's visualise the cosine
similarity between all CIFAR-10 class text embeddings — classes that are semantically
similar (e.g., *cat* and *dog*) should cluster together.

In [ ]:
# Text-text similarity matrix
sim_matrix = (text_embs @ text_embs.T).cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR_CLASSES, rotation=45, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CIFAR_CLASSES)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center",
                fontsize=8, color="black" if sim_matrix[i,j] < 0.7 else "white")
plt.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title("CLIP text embedding similarity between CIFAR-10 classes", fontsize=12)
plt.tight_layout(); plt.show()

### Final Summary

| Method | Dataset | # Labels | Test Accuracy |
|---|---|---|---|
| ViT from scratch | MNIST | 60k | ~98 % |
| ViT from scratch | Fashion-MNIST | 60k | ~90 % |
| ViT from scratch | CIFAR-10 | 50k | ~75–80 % |
| Pre-trained ViT (HF) | CIFAR-10 | 5k | ~90–93 % |
| OpenCLIP zero-shot | CIFAR-10 | **0** | ~76–80 % |

Key takeaways:
1. **ViT needs data** — from-scratch training on small datasets is hard without inductive biases.
2. **Pre-training is powerful** — 5k labels + ImageNet-21k pre-training beats 50k labels from scratch.
3. **Zero-shot CLIP** matches from-scratch ViT with *no labels at all*, using only natural language.
4. **Attention maps** show that ViT genuinely learns meaningful spatial structure, not just global statistics.
5. **Prompt engineering** matters for CLIP — carefully designed prompts can meaningfully shift accuracy.

### Further Reading

- [An Image is Worth 16x16 Words (ViT)](https://arxiv.org/abs/2010.11929) — Dosovitskiy et al., 2020
- [Learning Transferable Visual Models From Natural Language Supervision (CLIP)](https://arxiv.org/abs/2103.00020) — Radford et al., 2021
- [Attention Rollout](https://arxiv.org/abs/2005.00928) — Abnar & Zuidema, 2020
- [How Do Vision Transformers Work?](https://arxiv.org/abs/2202.06709) — Park & Kim, 2022
- [OpenCLIP](https://github.com/mlfoundations/open_clip) — open-source CLIP
- [DINOv2](https://arxiv.org/abs/2304.07193) — Meta's self-supervised ViT (excellent for fine-tuning)
- [TIPS: Text-Image Pretraining with Spatial awareness](https://arxiv.org/abs/2410.16512) — Google, 2024